Chapter 2 of Victor Dibia's [Designing Multi-Agent Systems](https://multiagentbook.com/) compares ways to decide which agent acts next. A workflow follows routes defined by the developer. A plan-based system can revise its plan as work comes back, and a group chat can use a model to choose the next speaker.

These notes focus on how those patterns pass control and context between agents. Chapters 6 and 7 return to their implementation.


## Who chooses the next step?

Both families draw on ideas from workflow engines such as Airflow, Temporal, and Prefect. They need to track dependencies and share state between steps. The difference is how much of the route can be set before the run.

A workflow can branch on a condition the developer wrote. An autonomous system can ask a model to interpret a result and decide what to do next. That helps when a result is hard to classify in advance, but the model can make different decisions on similar inputs. Each decision also consumes tokens. A design that gives the model more control needs a way to detect and recover from more kinds of errors.


## Workflow patterns

A workflow represents tasks as nodes in a graph. A node can run a function, call an agent, or execute a nested workflow. Edges determine where the output goes. An edge can depend on the previous result, shared state, or an external signal. Because the routes are defined before execution, the graph can be checked before a run.

- **Sequential.** Each step passes its output to the next. The book's news pipeline starts by selecting topics and ends by writing a report. If the report misses a story, the intermediate results show where the story dropped out.
- **Conditional.** A rule chooses among routes. Failed tests can send work back to a coding agent. A supervisor can classify an incoming request and route it to a specialist. A node can also run a whole workflow of its own.
- **Parallel.** Independent branches run at the same time. A later step waits for their results and combines them.

These patterns work when the steps and dependencies are known well enough to specify and test. An unfamiliar task may require the system to decide how to divide the work after seeing intermediate results.


## Autonomous patterns

**Plan-based orchestration.** An orchestrator writes a plan, assigns steps, and tracks progress. It can give each worker only the context needed for its assignment. Magentic-One tracks the task and its progress in separate ledgers. Anthropic's research system starts parallel subagents as needed. In the book's `picoagents` example, the orchestrator evaluates each step and includes its failure analysis when retrying. The design gives one agent a view of the whole task, so a planning error can affect every worker.

**Handoff.** An agent decides when to transfer control to a peer and passes along the state the peer needs. In the implementation described in the book, a handoff is exposed as a tool call. The design distributes routing decisions across the agents. It also makes it easier for two agents to keep passing the task back and forth unless the system tracks progress across handoffs.

**Group chat.** Agents take turns in a shared conversation. A round-robin policy uses a fixed order, while a model can choose the next speaker from the conversation so far. Dibia relates this structure to single-agent techniques such as ReAct, which alternates reasoning and action, and Reflexion, which carries feedback into later attempts.

A shared conversation gets longer with each turn, which can raise token costs and eventually exceed the context window. The chapter also cites Cognition's observation that agents can make conflicting decisions even when they see the same messages. The messages may omit the assumptions behind a decision.
